# Proyek 1: Prediksi Keterlambatan Pengiriman - Olist E-Commerce
## Notebook 1: Data Cleaning & Validation 

### **1. Background Bisnis & Problem Statement**
Olist menghadapi tantangan besar dalam menjaga ketepatan waktu pengiriman. Dalam ekosistem e-commerce, keterlambatan bukan sekadar masalah logistik, melainkan masalah kepercayaan pelanggan yang berdampak langsung pada retensi.

**Tujuan Proyek**: Membangun model *machine learning* untuk memprediksi risiko keterlambatan pengiriman (`is_late`). Dengan prediksi ini, Olist dapat mengambil tindakan preventif sebelum keterlambatan benar-benar terjadi.

---

### **2. Sumber Data & Metadata (3 Pilar Logistik)**

Untuk mencapai tujuan tersebut, kita secara strategis memilih **3 dataset utama** dari ekosistem Olist yaitu: **olist_orders_dataset.csv, olist_order_payments_dataset.csv dan olist_customers_dataset.csv** . Pemilihan ini didasarkan pada **3 Pilar Utama Logistik: Waktu, Jarak, dan Administrasi**, guna memastikan fitur prediktor yang digunakan adalah data yang sudah tersedia saat transaksi terjadi (*real-time*).

| Dataset | Deskripsi Isi | Kolom Kunci (Datetime) | Peran & Urgensi Bisnis |
| :--- | :--- | :--- | :--- |
| **Orders** | Jejak alur waktu pesanan (janji vs kenyataan). | `order_purchase_timestamp`, `order_estimated_delivery_date` | **Pilar Waktu**: Jantung analisis untuk mengonstruksi label target `is_late`. |
| **Customers** | Lokasi geografis pelanggan. | - | **Pilar Jarak**: Proksi rute kurir untuk membedakan profil risiko antar wilayah di Brasil. |
| **Payments** | Detail transaksi & skema cicilan. | - | **Pilar Administrasi**: Menangkap pengaruh verifikasi pembayaran terhadap kecepatan proses gudang. |

#### **Mengapa Membatasi pada 3 Dataset Ini?**
* **Menghindari Data Leakage**: Kita tidak menggunakan tabel seperti `reviews` karena ulasan baru muncul *setelah* barang sampai. Menggunakannya akan membuat model "menyontek" masa depan.
* **Efisiensi Prediksi**: Tabel lain seperti `products` memang relevan, namun untuk tahap awal, variabel lokasi, waktu, dan kecepatan administrasi memberikan sinyal paling kuat terhadap efisiensi logistik secara umum.

#### **Ketentuan & Standarisasi Tipe Data:**
* **Temporal (Datetime)**: Semua kolom tanggal wajib dikonversi ke format `datetime64` agar perhitungan durasi pengiriman dapat dilakukan secara matematis.
* **Kategorikal (Object)**: Kolom lokasi (`state`) dan jenis pembayaran akan diproses melalui teknik *Encoding* di tahap modeling.
* **Numerik (Float/Int)**: Nilai transaksi dan jumlah cicilan digunakan untuk menangkap kompleksitas administrasi pesanan.

---

### **3. Tujuan Notebook Ini**
Memastikan fondasi data kokoh melalui dua langkah utama:
1. **Integritas Waktu**: Menstandarisasi format tanggal agar perhitungan durasi pengiriman akurat. 
2. **Kualitas Data**: Menangani *missing values* dan data duplikat pada 3 dataset di atas agar model tidak belajar dari data yang salah (*bias*).

### Berikut adalah langkah-langkah Data Cleaning & Validation

### 1. **Data Ingestion**: 
Memuat dataset dari folder `raw` untuk menjaga kemurnian data asli (Prinsip Data Pipeline yang baik).

In [1]:
import pandas as pd
import os

# Opsional: biar tampilan tabelnya lengkap kalau datanya banyak
pd.set_option('display.max_columns', None)

def load_data(file_path):
    """
    Fungsi untuk memuat dataset dari file CSV.
    
    Args:
        file_path (str): Alamat lokasi file CSV.
        
    Returns:
        df (pd.DataFrame): Dataframe yang sudah dimuat.
    """
    df = pd.read_csv(file_path)
    return df

# Eksekusi fungsi
df_orders = load_data('../data/raw/olist_orders_dataset.csv')
df_customers = load_data('../data/raw/olist_customers_dataset.csv')
df_payments = load_data('../data/raw/olist_order_payments_dataset.csv')

print("Data berhasil dimuat!")

Data berhasil dimuat!


### Langkah 1.1: Preview Data
Melakukan inspeksi visual awal untuk memahami struktur kolom dan tipe data pada dataset.

In [2]:
def preview_data(df, df_name):
    """
    Menampilkan dimensi data (shape), 5 data teratas, dan informasi tipe data.
    
    Args:
        df (pd.DataFrame): Dataframe yang ingin diintip.
        df_name (str): Nama dataframe.
    """
    print(f"--- Dataset: {df_name} ---")
    
    # Menambahkan pengecekan dimensi (Shape)
    print(f"Dimensi Data (Baris, Kolom): {df.shape}")
    
    print(f"\n--- 5 Data Teratas ---")
    display(df.head()) 
    
    print(f"\n--- Info Tipe Data ---")
    df.info()
    print("\n" + "="*50 + "\n")

# Eksekusi fungsi untuk kenalan
preview_data(df_orders, "Orders")
preview_data(df_customers, "Customers") 
preview_data(df_payments, "Payments")

--- Dataset: Orders ---
Dimensi Data (Baris, Kolom): (99441, 8)

--- 5 Data Teratas ---


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00



--- Info Tipe Data ---
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


--- Dataset: Customers ---
Dimensi Data (Baris, Kolom): (99441, 5)

--- 5 Data Teratas ---


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP



--- Info Tipe Data ---
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


--- Dataset: Payments ---
Dimensi Data (Baris, Kolom): (103886, 5)

--- 5 Data Teratas ---


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45



--- Info Tipe Data ---
<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 MB




#### Insight:
Dataset ini mencakup 99.441 pesanan unik. Terdapat 103.886 catatan pembayaran, di mana perbedaan jumlah ini disebabkan oleh adanya beberapa metode pembayaran yang digunakan dalam satu pesanan tunggal (split payment).

### 2. **Data Validation**: Memeriksa dimensi (`shape`), tipe data, nilai yang hilang (`missing values`), dan data duplikat.

In [3]:
def check_and_drop_duplicates(df, df_name):
    """
    Menghitung jumlah data duplikat dan menghapusnya jika ditemukan.
    
    Args:
        df (pd.DataFrame): Dataframe yang ingin dicek.
        df_name (str): Nama dataframe untuk tampilan print.
        
    Returns:
        df (pd.DataFrame): Dataframe yang sudah bersih dari duplikat.
    """
    duplicate_count = df.duplicated().sum()
    print(f"Jumlah duplikat di {df_name}: {duplicate_count}")
    
    if duplicate_count > 0:
        df = df.drop_duplicates()
        print(f"Selesai menghapus duplikat di {df_name}.")
    
    return df

# Eksekusi fungsi untuk masing-masing data
df_orders = check_and_drop_duplicates(df_orders, "Orders")
df_customers = check_and_drop_duplicates(df_customers, "Customers")
df_payments = check_and_drop_duplicates(df_payments, "Payments")

Jumlah duplikat di Orders: 0
Jumlah duplikat di Customers: 0
Jumlah duplikat di Payments: 0


#### Insight:
Hasil validasi menunjukkan tidak terdapat data duplikat pada ketiga dataset utama. Hal ini menandakan bahwa setiap baris data bersifat unik dan integritas data terjaga.

### 3. **Data Cleaning**: Memperbaiki tipe data (terutama kolom tanggal) dan menangani data yang tidak konsisten.

#### 3.1. Konversi Tipe Data (Handling Datetime)
Mengubah tipe data kolom tanggal dari string menjadi datetime untuk memungkinkan analisis berbasis waktu. Standarisasi ke datetime diperlukan agar kita bisa melakukan operasi aritmatika (pengurangan) untuk mendapatkan durasi pengiriman di Notebook 3.

In [4]:
def convert_to_datetime(df, columns):
    """
    Mengubah daftar kolom tertentu menjadi tipe data datetime.
    
    Args:
        df (pd.DataFrame): Dataframe yang akan diolah.
        columns (list): Daftar nama kolom yang ingin diubah.
        
    Returns:
        df (pd.DataFrame): Dataframe dengan tipe data yang sudah diperbarui.
    """
    for col in columns:
        df[col] = pd.to_datetime(df[col])
    return df

# Daftar kolom tanggal di dataset Orders
date_cols = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

# Eksekusi fungsi
df_orders = convert_to_datetime(df_orders, date_cols)

# Cek hasil konversi
print("Tipe data setelah dikonversi:")
print(df_orders[date_cols].dtypes)

Tipe data setelah dikonversi:
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


#### 3.2. Standardisasi Teks (Lowercase)

In [5]:
def lowercase_columns(df, columns):
    """
    Mengubah isi kolom kategori menjadi huruf kecil agar konsisten.
    """
    for col in columns:
        df[col] = df[col].str.lower()
    return df

# Eksekusi untuk kolom kategori yang penting
df_orders = lowercase_columns(df_orders, ['order_status'])
df_payments = lowercase_columns(df_payments, ['payment_type'])

# Tampilkan output
print("Hasil pengecekan status pesanan (setelah lowercase):")
print(df_orders['order_status'].unique())

print("\nHasil pengecekan tipe pembayaran (setelah lowercase):")
print(df_payments['payment_type'].unique())

Hasil pengecekan status pesanan (setelah lowercase):
<StringArray>
[  'delivered',    'invoiced',     'shipped',  'processing', 'unavailable',
    'canceled',     'created',    'approved']
Length: 8, dtype: str

Hasil pengecekan tipe pembayaran (setelah lowercase):
<StringArray>
['credit_card', 'boleto', 'voucher', 'debit_card', 'not_defined']
Length: 5, dtype: str


#### 3.3 Handling Missing Values
Kita buat fungsi khusus untuk mengisi data yang kosong agar tetap konsisten. Kita membuang data pesanan yang tidak memiliki tanggal pengiriman (delivered_customer_date) karena tanpa data ini, kita tidak bisa melabeli apakah pesanan tersebut telat atau tidak untuk melatih model.

In [6]:
def handle_missing_values(df, column, fill_value="others"):
    """
    Mengisi data yang hilang (NaN) pada kolom tertentu dengan nilai default.
    
    Args:
        df (pd.DataFrame): Dataframe yang akan diproses.
        column (str): Nama kolom yang akan diisi missing value-nya.
        fill_value (str): Nilai pengganti untuk NaN.
    """
    if column in df.columns:
        df[column] = df[column].fillna(fill_value)
        print(f"Berhasil: Missing value di kolom '{column}' diisi dengan '{fill_value}'.")
    else:
        print(f"Info: Kolom '{column}' tidak ditemukan di dataframe ini. Langkah dilewati.")
    return df

# Eksekusi pengecekan
df_orders = handle_missing_values(df_orders, 'product_category_name')

# Verifikasi jumlah baris
print(f"Jumlah data akhir: {df_orders.shape[0]} baris")

Info: Kolom 'product_category_name' tidak ditemukan di dataframe ini. Langkah dilewati.
Jumlah data akhir: 99441 baris


### Langkah 4: Data Export
Menyimpan data hasil pembersihan ke folder interim dalam format pickle (.pkl) untuk menjaga konsistensi tipe data.

In [7]:
import os

def export_to_interim(df, file_name):
    """
    Menyimpan dataframe ke folder interim dalam format pickle (.pkl).
    """
    # Menentukan path (asumsi folder data sejajar dengan folder notebook)
    output_dir = "../data/interim/"
    
    # Pastikan folder sudah ada, kalau belum kita buat otomatis
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    path = f"{output_dir}{file_name}.pkl"
    df.to_pickle(path)
    print(f"Berhasil ekspor: {path}")

# --- EKSEKUSI EKSPOR ---
# Kita simpan semua tabel yang sudah dibersihkan tadi
export_to_interim(df_orders, "orders_clean")
export_to_interim(df_payments, "payments_clean")
export_to_interim(df_customers, "customers_clean")

# Jika ada df_merge (hasil gabungan), jangan lupa diekspor juga!
# export_to_interim(df_merge, "01_data_cleaned")

print("\n--- PROSES CLEANING SELESAI: DATA SIAP UNTUK EDA! ---")

Berhasil ekspor: ../data/interim/orders_clean.pkl
Berhasil ekspor: ../data/interim/payments_clean.pkl
Berhasil ekspor: ../data/interim/customers_clean.pkl

--- PROSES CLEANING SELESAI: DATA SIAP UNTUK EDA! ---


#### 4.1. Final Verification (Validasi Ekspor)
Gunakan modul os untuk mengintip isi folder interim dan memastikan ukuran filenya tidak nol (artinya ada isinya).

In [8]:
def verify_final_data(df):
    """
    Melakukan verifikasi integritas data pada memori sebelum proses selesai.
    
    Args:
        df (pd.DataFrame): Dataframe yang ingin dicek jumlah baris dan kolomnya.
    """
    shape = df.shape
    print(f"--- 1. Verifikasi Isi Dataframe ---")
    print(f"Jumlah Baris: {shape[0]}")
    print(f"Jumlah Kolom: {shape[1]}")
    
    # Verifikasi angka 99.441 sesuai dataset awal Olist
    if shape[0] == 99441:
        print("Status: DATA INTEGRITY AMAN (Sesuai dengan record awal)")
    else:
        print(f"Status: Ada perubahan jumlah data (Sekarang: {shape[0]})")

def verify_exports(directory):
    """
    Memastikan file hasil ekspor (.pkl) tersedia di folder tujuan dan mencatat ukurannya.
    
    Args:
        directory (str): Path folder tempat penyimpanan file interim.
    """
    import os
    print(f"\n--- 2. Memverifikasi Folder Ekspor: {directory} ---")
    
    if os.path.exists(directory):
        files = os.listdir(directory)
        if not files:
            print("Folder kosong! Ekspor kemungkinan gagal.")
        else:
            for file in files:
                file_path = os.path.join(directory, file)
                file_size = os.path.getsize(file_path) / 1024  # Konversi ke KB
                print(f"Terdeteksi: {file} | Ukuran: {file_size:.2f} KB")
    else:
        print("Folder tidak ditemukan! Pastikan path folder sudah benar.")

# --- Eksekusi Pengecekan Terakhir ---
verify_final_data(df_orders)
verify_exports("../data/interim/")

print("\n--- SEMUA TAHAP SELESAI: DATA SIAP UNTUK EDA! ---")

--- 1. Verifikasi Isi Dataframe ---
Jumlah Baris: 99441
Jumlah Kolom: 8
Status: DATA INTEGRITY AMAN (Sesuai dengan record awal)

--- 2. Memverifikasi Folder Ekspor: ../data/interim/ ---
Terdeteksi: payments_clean.pkl | Ukuran: 7157.22 KB
Terdeteksi: customers_clean.pkl | Ukuran: 8488.19 KB
Terdeteksi: orders_clean.pkl | Ukuran: 11849.63 KB

--- SEMUA TAHAP SELESAI: DATA SIAP UNTUK EDA! ---


### Kesimpulan Tahap 1
Proses pembersihan data telah selesai dilakukan pada Orders, Customers, dan Payments secara terpisah (atomik) untuk menjaga integritas masing-masing tabel.

**Hasil Utama**:

**Data Valid**: Kolom temporal (tanggal) sudah dalam format datetime.

**Data Siap Gabung**: Dataset sudah bersih dari duplikat dan siap untuk dikonsolidasikan.

**Penyimpanan**: Data disimpan dalam folder ../data/interim/ dalam format .pkl untuk mempertahankan tipe data yang sudah diperbaiki.

Next Step: Kita akan beralih ke Notebook 2: Exploratory Data Analysis (EDA) untuk menggabungkan dataset ini dan mulai mencari pola hubungan antara lokasi, pembayaran, dan keterlambatan pengiriman.